# MiniMax-H3 · 模块与训练直觉（配套 Notebook）

> 配套长文：[MiniMax-H3模型构成与训练全流程.md](./MiniMax-H3模型构成与训练全流程.md)
> 定位：把 VAE token 账、AdaLN 13B、Rectified Flow、MM-RoPE、flow_shift 全部跑成数字。纯 Python 标准库，零依赖。

**怎么跑**：逐格 `Shift+Enter`。

| 本 Notebook | 长文章节 | 验证什么 |
|---|---|---|
| §1 视频 / 音频 token | §3.2 / §3.3 | 10 秒 16:9 为什么是 60480 个视频 token |
| §2 AdaLN 参数账 | §4.1 | 为什么约 13B 可以卸掉 |
| §3 一维 Rectified Flow | §5 | 直线插值 + MSE 速度 + Euler 采样 |
| §4 MM-RoPE 旋转 | §3.4 | 相对 (Δt,Δh) 相同则点积相同 |
| §5 flow_shift | §5.2 | 开源数字 12 vs 3；拧向哪侧是工作模型 |


## 1. 10 秒片子有多少 token？（长文 §3）

VisualVAE：`f16t4d24`，再 `1×2×2` patch。AudioVAE：32 kHz → 40 Hz。


In [ ]:
seconds = 10
fps = 24
height, width = 768, 1344  # 16:9，短边 768

frames = seconds * fps
t_lat = frames // 4
h_lat = height // 16
w_lat = width // 16
h_tok = h_lat // 2
w_tok = w_lat // 2
video_tokens = t_lat * h_tok * w_tok
audio_tokens_per_ch = seconds * 40
video_in_dim = 24 * 1 * 2 * 2  # patch 后每个视频 token

print(f"像素帧 {frames}")
print(f"VAE 网格 (T,H,W) = ({t_lat}, {h_lat}, {w_lat})")
print(f"patch 后 (T,H,W) = ({t_lat}, {h_tok}, {w_tok})")
print(f"视频 token = {video_tokens:,}  （每 token 输入维 {video_in_dim}）")
print(f"音频 token / 声道 = {audio_tokens_per_ch}  （立体声 packed 可能翻倍）")
print(f"视频:每声道音频 ≈ {video_tokens / audio_tokens_per_ch:.0f} : 1  （画面才是序列大头）")
assert video_tokens == 60480


## 2. AdaLN 为什么是 ~13B？（长文 §4.1）

每层：时间嵌入 2688 维 → 3 个模态 × 6 个向量（attn/mlp 的 shift、scale、gate）× hidden 5376。


In [ ]:
layers = 50
time_embed_dim = 2688
hidden = 5376
expand = 6
modalities = 3

adaln_out = expand * hidden * modalities  # 18 * 5376 = 96768
per_layer = time_embed_dim * adaln_out + adaln_out  # 含 bias
total = layers * per_layer

print(f"每层 AdaLN 输出维 = {adaln_out:,}")
print(f"每层参数 ≈ {per_layer/1e8:.2f}×10⁸")
print(f"50 层 ≈ {total/1e9:.2f} B   （官方口径约 13B）")
assert abs(total / 1e9 - 13.0) < 0.1
print("推理时同一 t 只算一次调制，这 13B 可以不驻留。微调必须留着。")


## 3. 一维 Rectified Flow（长文 §5）

真值是长度为 8 的小波。网络是两层线性（手写 SGD）。目标：在直线 \(x_t=(1-t)x_0+t\varepsilon\) 上预测速度 \(v=\varepsilon-x_0\)。


In [ ]:
import math
import random

random.seed(0)

N = 8
x0 = [math.sin(2 * math.pi * i / N) for i in range(N)]
HID = 32


def zeros(r, c=None):
    if c is None:
        return [0.0] * r
    return [[0.0] * c for _ in range(r)]


def randn_mat(r, c, scale):
    return [[random.gauss(0, scale) for _ in range(c)] for _ in range(r)]


W1 = randn_mat(HID, N + 1, 0.2)  # concat(x_t, t)
b1 = zeros(HID)
W2 = randn_mat(N, HID, 0.2)
b2 = zeros(N)


def relu(xs):
    return [x if x > 0 else 0.0 for x in xs]


def matvec(W, x):
    return [sum(wi * xj for wi, xj in zip(row, x)) for row in W]


def forward(xt, t):
    inp = xt + [t]
    h = relu([u + b for u, b in zip(matvec(W1, inp), b1)])
    out = [u + b for u, b in zip(matvec(W2, h), b2)]
    return out, h, inp


def train(steps=800, lr=0.05):
    last = 0.0
    for step in range(steps):
        eps = [random.gauss(0, 1) for _ in range(N)]
        t = random.random()
        xt = [(1 - t) * a + t * e for a, e in zip(x0, eps)]
        v = [e - a for a, e in zip(x0, eps)]
        pred, h, inp = forward(xt, t)
        err = [p - t_true for p, t_true in zip(pred, v)]
        last = sum(e * e for e in err) / N

        # dL/dpred = 2 err / N
        g_out = [2 * e / N for e in err]
        g_h = zeros(HID)
        for i in range(N):
            for j in range(HID):
                g_h[j] += g_out[i] * W2[i][j]
                W2[i][j] -= lr * g_out[i] * h[j]
            b2[i] -= lr * g_out[i]
        g_h = [g if h[j] > 0 else 0.0 for j, g in enumerate(g_h)]
        for j in range(HID):
            for k in range(N + 1):
                W1[j][k] -= lr * g_h[j] * inp[k]
            b1[j] -= lr * g_h[j]
        if (step + 1) % 200 == 0:
            print(f"step {step+1:4d}  mse={last:.4f}")
    return last


mse = train()
print(f"最终 mse={mse:.4f}")


In [ ]:
# Euler：从纯噪声 x=ε 沿预测速度往回走到 t=0（这里 v=ε-x0，dx/dt=v，所以 dt 为负）
random.seed(1)
x = [random.gauss(0, 1) for _ in range(N)]
n_steps = 20
dt = 1.0 / n_steps
t = 1.0
for _ in range(n_steps):
    v_hat, _, _ = forward(x, t)
    x = [xi - dt * vi for xi, vi in zip(x, v_hat)]  # t: 1→0
    t -= dt

def fmt(xs):
    return " ".join(f"{v:+.2f}" for v in xs)

print("真值 x0 ", fmt(x0))
print("采样结果", fmt(x))
cos = sum(a * b for a, b in zip(x0, x))
n1 = math.sqrt(sum(a * a for a in x0))
n2 = math.sqrt(sum(a * a for a in x))
print(f"余弦相似度 {cos / (n1 * n2):.3f}  （玩具网，能看到波形对上即可）")


## 4. MM-RoPE：相对位移相同则分数相同（长文 §3.4）

二维玩具：只转 `(t, h)` 两个平面。同一相对位移 `(Δt, Δh)` 的点积应相等。


In [ ]:
import math


def rotate(vec2, theta):
    x, y = vec2
    c, s = math.cos(theta), math.sin(theta)
    return (c * x - s * y, s * x + c * y)


def mm_rope(q, k, pos_q, pos_k, base=10000.0):
    """q,k 各 4 维：前两维转 t，后两维转 h。"""
    tq, hq = pos_q
    tk, hk = pos_k
    theta_t = 1.0 / base
    theta_h = 1.0 / (base ** 0.5)
    qr = rotate(q[:2], tq * theta_t) + rotate(q[2:], hq * theta_h)
    kr = rotate(k[:2], tk * theta_t) + rotate(k[2:], hk * theta_h)
    return sum(a * b for a, b in zip(qr, kr))


q = (1.0, 0.2, 0.5, -0.3)
k = (0.4, 0.8, -0.1, 0.7)
s1 = mm_rope(q, k, (0, 0), (2, 1))
s2 = mm_rope(q, k, (5, 3), (7, 4))  # 同样 Δt=2, Δh=1
s3 = mm_rope(q, k, (0, 0), (2, 2))  # Δh 不同
print(f"相对 (2,1) @ 原点   {s1:.6f}")
print(f"相对 (2,1) @ (5,3)  {s2:.6f}")
print(f"相对 (2,2)          {s3:.6f}")
assert abs(s1 - s2) < 1e-9
print("相对位移相同 → 分数相同。H3 再多一个 w 轴，音频主要走 t 轴。")


## 5. flow_shift（长文 §5.2）

数字来自开源请求体。下面公式与 SD3 / Flux 同类，是工作模型，官方未写：

\[
t' = \frac{s\,t}{1+(s-1)t}
\]

H3 视频 `s=12`、音频 `s=3`。看均匀 t 被映射到哪里。


In [ ]:
def shift_t(t, s):
    return (s * t) / (1 + (s - 1) * t)


ts = [i / 10 for i in range(11)]
print("  t     shift=1   音频s=3   视频s=12")
for t in ts:
    print(f"{t:4.1f}    {shift_t(t,1):6.3f}    {shift_t(t,3):6.3f}     {shift_t(t,12):6.3f}")
print()
print("工作模型（本文 t=1 为噪声）：s 越大越拧向 1。方向取决于调度器记号，不是官方公式。")
print("开源请求体：视频 flow_shift=12，音频 audio_flow_shift=3。")
